In [3]:
import gc
from pathlib import Path
import pandas as pd
from tqdm.auto import tqdm
 
# =========================================================
# 0. CONFIGURATION
# =========================================================

DATA_DIR = Path("../data/mimic-iv")
 
# =========================================================
# 1. AUDIT THE TARGET COHORT'S INDEX VISIT
# =========================================================

print("--- Step 1: Auditing Target Cohort Index Encounters ---")
target_macro = pd.read_csv("../data/intermediate/aortic_dissection_macro_visits.csv")
target_macro["encounter_time"] = pd.to_datetime(target_macro["encounter_time"])

# Identify visits that contained the diagnosis
target_macro["is_dissection_visit"] = target_macro["diagnosis"].fillna("").str.contains(r"441|I71", regex=True)
dissection_visits = target_macro[target_macro["is_dissection_visit"]].copy()
 
# Isolate the very first visit (Index Encounter) per target patient
target_index = dissection_visits.sort_values(by=["subject_id", "encounter_time"]).groupby("subject_id").first().reset_index()
 
# Calculate ED vs Non-ED origins
target_index["is_ed_origin"] = (
    (target_index["admission_location"] == "EMERGENCY ROOM") | 
    (target_index["encounter_urgency"] == "STANDALONE_ED")
)
 
ed_count = target_index["is_ed_origin"].sum()
total_targets = len(target_index)

print(f"Target Patients: {total_targets:,}")
print(f"Index Visits originating in ED: {ed_count:,} ({(ed_count/total_targets)*100:.1f}%)")
print(f"Index Visits NOT originating in ED: {total_targets - ed_count:,} ({(1 - ed_count/total_targets)*100:.1f}%)\n")
 
# Store target IDs for exclusion
target_ids = set(target_index["subject_id"].unique())
 
# =========================================================
# 2. DEFINE THE CONTROL UNIVERSE
# =========================================================

print("--- Step 2: Defining Control Universe ---")
all_patients = pd.read_csv(DATA_DIR / "hosp" / "patients.csv", usecols=["subject_id"])
control_ids = set(all_patients["subject_id"].unique()) - target_ids

print(f"Total Control Patients Available: {len(control_ids):,}\n")
del all_patients
 
# =========================================================
# 3. GLOBAL ED SWEEP FOR CONTROLS
# =========================================================

print("--- Step 3: Sweeping MIMIC-IV for Control ED Visits ---")
control_ed_shards = []
 
# A. Sweep Formal Inpatient Admissions originating in the ED
adm_cols = ["subject_id", "hadm_id", "admittime", "admission_location"]

for chunk in pd.read_csv(DATA_DIR / "hosp" / "admissions.csv", usecols=adm_cols, chunksize=500_000):
    # Filter to controls AND Emergency Room locations
    hit = chunk[
        (chunk["subject_id"].isin(control_ids)) & 
        (chunk["admission_location"] == "EMERGENCY ROOM")
    ].copy()
    if not hit.empty:
        hit["encounter_time"] = pd.to_datetime(hit["admittime"])
        hit["ed_type"] = "ADMITTED_ED"
        control_ed_shards.append(hit[["subject_id", "hadm_id", "encounter_time", "ed_type"]])
 
# B. Sweep Transfers for Standalone (Discharged) ED Visits
trn_cols = ["subject_id", "hadm_id", "careunit", "intime"]

for chunk in pd.read_csv(DATA_DIR / "hosp" / "transfers.csv", usecols=trn_cols, chunksize=1_000_000):
    # Filter to controls, Emergency Department careunit, and NULL hadm_id (standalone)
    hit = chunk[
        (chunk["subject_id"].isin(control_ids)) & 
        (chunk["careunit"] == "Emergency Department") & 
        (chunk["hadm_id"].isna())
    ].copy()
    if not hit.empty:
        hit["encounter_time"] = pd.to_datetime(hit["intime"])
        hit["ed_type"] = "STANDALONE_ED"

        # Generate the exact same synthetic key structure used for targets
        hit["hadm_id"] = "ED_" + hit["subject_id"].astype(str) + "_" + hit["encounter_time"].dt.strftime("%Y%m%d%H%M")
        control_ed_shards.append(hit[["subject_id", "hadm_id", "encounter_time", "ed_type"]])
 
all_control_ed_visits = pd.concat(control_ed_shards, ignore_index=True)

del control_ed_shards
gc.collect()
print(f"Found {len(all_control_ed_visits):,} total ED visits across the control universe.\n")
 
# =========================================================
# 4. ISOLATE THE MOST RECENT ED VISIT PER CONTROL
# =========================================================

print("--- Step 4: Isolating 'Most Recent' Visit ---")

# Sort descending by time, then drop duplicates to keep only the absolute latest visit per subject
control_index = all_control_ed_visits.sort_values(
    by=["subject_id", "encounter_time"], ascending=[True, False]
).drop_duplicates(subset=["subject_id"], keep="first")
 
# Rename columns to match target baseline expectations
control_index.rename(columns={"encounter_time": "index_time", "hadm_id": "index_hadm_id"}, inplace=True)
 
# Export the master control cohort tracker
output_path = "control_ed_index.csv"
control_index.to_csv(output_path, index=False)
print(f"Final Control Cohort Size: {len(control_index):,} unique patients.")
print(f"✔ Initial Control Index saved to: {output_path}")
 

--- Step 1: Auditing Target Cohort Index Encounters ---
Target Patients: 622
Index Visits originating in ED: 279 (44.9%)
Index Visits NOT originating in ED: 343 (55.1%)

--- Step 2: Defining Control Universe ---
Total Control Patients Available: 364,005

--- Step 3: Sweeping MIMIC-IV for Control ED Visits ---
Found 651,748 total ED visits across the control universe.

--- Step 4: Isolating 'Most Recent' Visit ---
Final Control Cohort Size: 297,706 unique patients.
✔ Initial Control Index saved to: control_ed_index.csv


In [5]:
import pandas as pd
 
# =========================================================
# 1. LOAD AND ISOLATE INDEX DIAGNOSIS VISITS
# =========================================================

target_macro = pd.read_csv("../data/intermediate/aortic_dissection_macro_visits.csv")
target_macro["encounter_time"] = pd.to_datetime(target_macro["encounter_time"])
 
# Isolate visits containing the specific billing codes
target_macro["is_dissection_visit"] = target_macro["diagnosis"].fillna("").str.contains(r"441|I71", regex=True)
dissection_visits = target_macro[target_macro["is_dissection_visit"]].copy()
 
# Lock onto the very first acute presentation per patient
target_index = dissection_visits.sort_values(
    by=["subject_id", "encounter_time"]
).groupby("subject_id").first().reset_index()
 
# =========================================================
# 2. GENERATE THOROUGH FREQUENCY TABLES
# =========================================================

print(f"Total Target Patients Assessed: {len(target_index):,}\n")
print("=== TABLE A: Admission Location (Physical Origin) ===")

loc_freq = target_index['admission_location'].value_counts(dropna=False).reset_index()
loc_freq.columns = ['Admission Location', 'Patient Count']
loc_freq['Percentage'] = (loc_freq['Patient Count'] / len(target_index)) * 100

print(loc_freq.to_string(index=False, formatters={'Percentage': '{:.1f}%'.format})) 
print("\n=== TABLE B: Admission Urgency (Clinical Context) ===")

urg_freq = target_index['encounter_urgency'].value_counts(dropna=False).reset_index()
urg_freq.columns = ['Encounter Urgency', 'Patient Count']
urg_freq['Percentage'] = (urg_freq['Patient Count'] / len(target_index)) * 100

print(urg_freq.to_string(index=False, formatters={'Percentage': '{:.1f}%'.format}))
 

Total Target Patients Assessed: 622

=== TABLE A: Admission Location (Physical Origin) ===
                    Admission Location  Patient Count Percentage
                        EMERGENCY ROOM            279      44.9%
                TRANSFER FROM HOSPITAL            185      29.7%
                    PHYSICIAN REFERRAL            124      19.9%
                 WALK-IN/SELF REFERRAL             15       2.4%
                       CLINIC REFERRAL              7       1.1%
                                  PACU              4       0.6%
                        PROCEDURE SITE              3       0.5%
             INFORMATION NOT AVAILABLE              3       0.5%
TRANSFER FROM SKILLED NURSING FACILITY              2       0.3%

=== TABLE B: Admission Urgency (Clinical Context) ===
          Encounter Urgency  Patient Count Percentage
                   EW EMER.            355      57.1%
                     URGENT             82      13.2%
          OBSERVATION ADMIT             76

In [6]:
import gc
from pathlib import Path
import pandas as pd
 
# =========================================================
# 0. CONFIGURATION & COHORT DEFINITION
# =========================================================

DATA_DIR = Path("../data/mimic-iv")

# Load Target IDs to ensure strict exclusion
target_macro = pd.read_csv("../data/intermediate/aortic_dissection_macro_visits.csv")
target_ids = set(target_macro["subject_id"].unique())
 
# =========================================================
# 1. SWEEP ALL ADMISSIONS FOR CONTROLS
# =========================================================

print("Sweeping MIMIC-IV for all control patient admissions...")
adm_cols = ["subject_id", "hadm_id", "admittime", "admission_type", "admission_location"]
control_adms = []
 
for chunk in pd.read_csv(DATA_DIR / "hosp" / "admissions.csv", usecols=adm_cols, chunksize=500_000):
    # Exclude target patients
    hit = chunk[~chunk["subject_id"].isin(target_ids)].copy()
    if not hit.empty:
        control_adms.append(hit)

control_df = pd.concat(control_adms, ignore_index=True)
control_df["admittime"] = pd.to_datetime(control_df["admittime"])
del control_adms
gc.collect()
 
# =========================================================
# 2. ISOLATE MOST RECENT VISIT & CALCULATE FREQUENCIES
# =========================================================

print("Isolating the absolute most recent encounter per control patient...")
control_index = control_df.sort_values(
    by=["subject_id", "admittime"], ascending=[True, False]
).groupby("subject_id").first().reset_index()
 
print(f"\nTotal Control Patients Assessed: {len(control_index):,}\n")

print("=== TABLE A: Admission Location (Unfiltered Controls) ===")
loc_freq = control_index['admission_location'].value_counts(dropna=False).reset_index()
loc_freq.columns = ['Admission Location', 'Patient Count']
loc_freq['Percentage'] = (loc_freq['Patient Count'] / len(control_index)) * 100
print(loc_freq.to_string(index=False, formatters={'Percentage': '{:.1f}%'.format}))

print("\n=== TABLE B: Encounter Urgency (Unfiltered Controls) ===")
urg_freq = control_index['admission_type'].value_counts(dropna=False).reset_index()
urg_freq.columns = ['Encounter Urgency', 'Patient Count']
urg_freq['Percentage'] = (urg_freq['Patient Count'] / len(control_index)) * 100
print(urg_freq.to_string(index=False, formatters={'Percentage': '{:.1f}%'.format}))
 

Sweeping MIMIC-IV for all control patient admissions...
Isolating the absolute most recent encounter per control patient...

Total Control Patients Assessed: 222,830

=== TABLE A: Admission Location (Unfiltered Controls) ===
                    Admission Location  Patient Count Percentage
                        EMERGENCY ROOM          89414      40.1%
                    PHYSICIAN REFERRAL          66454      29.8%
                TRANSFER FROM HOSPITAL          32066      14.4%
                 WALK-IN/SELF REFERRAL          20709       9.3%
                       CLINIC REFERRAL           3876       1.7%
    INTERNAL TRANSFER TO OR FROM PSYCH           3060       1.4%
TRANSFER FROM SKILLED NURSING FACILITY           3056       1.4%
                        PROCEDURE SITE           2068       0.9%
                                  PACU           1757       0.8%
             INFORMATION NOT AVAILABLE            239       0.1%
           AMBULATORY SURGERY TRANSFER            130       

In [7]:
# REFINED CONTROL INDEX

import gc
from pathlib import Path
import pandas as pd
from tqdm.auto import tqdm
 
# =========================================================
# 0. CONFIGURATION & DEMOGRAPHICS LOAD
# =========================================================

DATA_DIR = Path("../data/mimic-iv")

print("Loading demographic baselines for age calculation...")
# Load patients to get anchor_age and anchor_year for calculating exact age at admission
pts = pd.read_csv(DATA_DIR / "hosp" / "patients.csv", usecols=["subject_id", "anchor_age", "anchor_year"])

# =========================================================
# 1. APPLY AGE FILTER TO TARGET COHORT
# =========================================================

print("\n--- Step 1: Refining Target Cohort (Age >= 18) ---")
target_macro = pd.read_csv("../data/intermediate/aortic_dissection_macro_visits.csv")
target_macro["encounter_time"] = pd.to_datetime(target_macro["encounter_time"])
 
# A. Isolate the exact visit containing the acute diagnosis
target_macro["is_dissection_visit"] = target_macro["diagnosis"].fillna("").str.contains(r"441|I71", regex=True)
dissection_visits = target_macro[target_macro["is_dissection_visit"]].copy()
 
# B. Lock onto the initial acute presentation
target_index = dissection_visits.sort_values(
    by=["subject_id", "encounter_time"]
).groupby("subject_id").first().reset_index()
 
# C. Calculate precise age at the time of the dissection
target_index = target_index.merge(pts, on="subject_id", how="inner")
target_index["index_age"] = target_index["anchor_age"] + (target_index["encounter_time"].dt.year - target_index["anchor_year"])
 
# D. Enforce Adult Barrier
initial_targets = len(target_index)
target_index = target_index[target_index["index_age"] >= 18].copy()
 
print(f"Dropped {initial_targets - len(target_index)} target patients under 18 years old.")
print(f"Final Adult Target Cohort: {len(target_index)} patients.")
 
# Store final target IDs for strict exclusion from controls
target_ids = set(target_index["subject_id"].unique())
 
# =========================================================
# 2. GLOBAL SWEEP FOR ALL CONTROL ENCOUNTERS
# =========================================================

print("\n--- Step 2: Sweeping MIMIC-IV for All Control Encounters ---")
control_shards = []
 
# A. Sweep Formal Inpatient Admissions (All locations/urgencies)
adm_cols = ["subject_id", "hadm_id", "admittime", "admission_type", "admission_location"]
for chunk in pd.read_csv(DATA_DIR / "hosp" / "admissions.csv", usecols=adm_cols, chunksize=500_000):
    hit = chunk[~chunk["subject_id"].isin(target_ids)].copy()
    if not hit.empty:
        hit["encounter_time"] = pd.to_datetime(hit["admittime"])
        hit["encounter_urgency"] = hit["admission_type"]
        control_shards.append(hit[["subject_id", "hadm_id", "encounter_time", "admission_location", "encounter_urgency"]])
 
# B. Sweep Transfers for Standalone Encounters (Null hadm_id, Any Unit)
trn_cols = ["subject_id", "hadm_id", "careunit", "intime"]
for chunk in pd.read_csv(DATA_DIR / "hosp" / "transfers.csv", usecols=trn_cols, chunksize=1_000_000):
    hit = chunk[(~chunk["subject_id"].isin(target_ids)) & (chunk["hadm_id"].isna())].copy()
    if not hit.empty:
        hit["encounter_time"] = pd.to_datetime(hit["intime"])
        hit["admission_location"] = hit["careunit"]  # Proxy location using the physical care unit
        hit["encounter_urgency"] = "STANDALONE_VISIT"
        # Generate deterministic synthetic keys for standalone visits
        hit["hadm_id"] = "SA_" + hit["subject_id"].astype(str) + "_" + hit["encounter_time"].dt.strftime("%Y%m%d%H%M")
        control_shards.append(hit[["subject_id", "hadm_id", "encounter_time", "admission_location", "encounter_urgency"]])
 
all_control_visits = pd.concat(control_shards, ignore_index=True)
del control_shards
gc.collect()
print(f"Found {len(all_control_visits):,} total hospital encounters across the control universe.")
 
# =========================================================
# 3. ISOLATE MOST RECENT VISIT & ENFORCE AGE
# =========================================================

print("\n--- Step 3: Isolating 'Most Recent' Visit & Enforcing Adult Age ---")
# Sort descending by time, drop duplicates to keep absolute latest visit per subject
control_index = all_control_visits.sort_values(
    by=["subject_id", "encounter_time"], ascending=[True, False]
).drop_duplicates(subset=["subject_id"], keep="first")

# Calculate precise age at this specific most recent encounter
control_index = control_index.merge(pts, on="subject_id", how="inner")
control_index["index_age"] = control_index["anchor_age"] + (control_index["encounter_time"].dt.year - control_index["anchor_year"])

# Enforce Adult Barrier
initial_controls = len(control_index)
control_index = control_index[control_index["index_age"] >= 18].copy()

print(f"Dropped {initial_controls - len(control_index):,} control patients under 18 years old.")
print(f"Final Adult Control Universe: {len(control_index):,} unique patients.\n")
 
# Cleanup and rename to match feature expectations
control_index.drop(columns=["anchor_age", "anchor_year"], inplace=True)
control_index.rename(columns={"encounter_time": "index_time", "hadm_id": "index_hadm_id"}, inplace=True)
 
# Export the master control cohort tracker
output_path = "control_master_index.csv"
control_index.to_csv(output_path, index=False)
print(f"✔ Master Control Index saved to: {output_path}")
 

Loading demographic baselines for age calculation...

--- Step 1: Refining Target Cohort (Age >= 18) ---
Dropped 0 target patients under 18 years old.
Final Adult Target Cohort: 622 patients.

--- Step 2: Sweeping MIMIC-IV for All Control Encounters ---
Found 952,337 total hospital encounters across the control universe.

--- Step 3: Isolating 'Most Recent' Visit & Enforcing Adult Age ---
Dropped 0 control patients under 18 years old.
Final Adult Control Universe: 364,004 unique patients.

✔ Master Control Index saved to: control_master_index.csv


In [9]:
import pandas as pd
from pathlib import Path
 
DATA_DIR = Path("../data/mimic-iv")
 
print("--- Step 1: Loading & Formatting the Adult Control Group ---")
controls = pd.read_csv("control_master_index.csv")
controls["index_time"] = pd.to_datetime(controls["index_time"])
controls["is_aortic_dissection"] = 0  # Negative Class Label
 
print("\n--- Step 2: Re-isolating the Adult Target Group ---")
# Load macro visits and demographic anchors
target_macro = pd.read_csv("../data/intermediate/aortic_dissection_macro_visits.csv")
target_macro["encounter_time"] = pd.to_datetime(target_macro["encounter_time"])
pts = pd.read_csv(DATA_DIR / "hosp" / "patients.csv", usecols=["subject_id", "anchor_age", "anchor_year"])
 
# Lock onto the exact dissection visit
target_macro["is_dissection_visit"] = target_macro["diagnosis"].fillna("").str.contains(r"441|I71", regex=True)
targets = target_macro[target_macro["is_dissection_visit"]].sort_values(
    by=["subject_id", "encounter_time"]
).groupby("subject_id").first().reset_index()
 
# Calculate age and enforce Adult Barrier
targets = targets.merge(pts, on="subject_id", how="inner")
targets["index_age"] = targets["anchor_age"] + (targets["encounter_time"].dt.year - targets["anchor_year"])
targets = targets[targets["index_age"] >= 18].copy()
 
# Rename to match Control feature space
targets.rename(columns={"encounter_time": "index_time", "hadm_id": "index_hadm_id"}, inplace=True)
targets["is_aortic_dissection"] = 1  # Positive Class Label
 
print("\n--- Step 3: Concatenating the ML Cohort Spine ---")
# Define the strict shared feature space
keep_cols = [
    "subject_id", 
    "index_hadm_id", 
    "index_time", 
    "index_age", 
    "admission_location", 
    "encounter_urgency", 
    "is_aortic_dissection"
]

# Vertically stack the cohorts
ml_spine = pd.concat([targets[keep_cols], controls[keep_cols]], ignore_index=True)

# Sort chronologically just to keep the matrix clean
ml_spine.sort_values(by=["index_time"], inplace=True)

output_path = "ml_cohort_spine.csv"
ml_spine.to_csv(output_path, index=False)

print(f"✔ Master ML Spine Exported: {len(ml_spine):,} total rows -> {output_path}")
print(f"  -> Targets (1): {ml_spine['is_aortic_dissection'].sum():,}")
print(f"  -> Controls (0): {len(ml_spine) - ml_spine['is_aortic_dissection'].sum():,}")
 

--- Step 1: Loading & Formatting the Adult Control Group ---

--- Step 2: Re-isolating the Adult Target Group ---

--- Step 3: Concatenating the ML Cohort Spine ---
✔ Master ML Spine Exported: 364,626 total rows -> ml_cohort_spine.csv
  -> Targets (1): 622
  -> Controls (0): 364,004


In [10]:
import gc
import pandas as pd
from pathlib import Path
 
# =========================================================
# 0. CONFIGURATION
# =========================================================

DATA_DIR = Path("../data/mimic-iv")
SPINE = pd.read_csv("ml_cohort_spine.csv")
SPINE["index_time"] = pd.to_datetime(SPINE["index_time"])
 
# Map IDs to human-readable names for the flat columns
vital_ids = {
    220045: "heart_rate",
    220050: "abp_sys", 220051: "abp_dias",
    220179: "nibp_sys", 220180: "nibp_dias",
    220210: "resp_rate", 220277: "spo2",
    223761: "temp_f", 223762: "temp_c"
}
 
# =========================================================
# 1. PROCESS VITALS IN CHUNKS
# =========================================================

print("Extracting and aggregating 24-hour Day-0 vitals...")
vital_list = []
 
for chunk in pd.read_csv(DATA_DIR / "icu" / "chartevents.csv", 
                         usecols=["subject_id", "charttime", "itemid", "valuenum"], 
                         chunksize=2_000_000):
    # Filter to patients in our cohort
    chunk = chunk[chunk["subject_id"].isin(SPINE["subject_id"])].copy()
    chunk["charttime"] = pd.to_datetime(chunk["charttime"])
    # Map to cohort index times
    chunk = chunk.merge(SPINE[["subject_id", "index_time"]], on="subject_id", how="inner")
    # Enforce Day-0 Window: (0 <= time_diff <= 24 hours)
    delta = (chunk["charttime"] - chunk["index_time"]).dt.total_seconds() / 3600
    chunk = chunk[(delta >= 0) & (delta <= 24)].copy()
    chunk["vital_name"] = chunk["itemid"].map(vital_ids)
    vital_list.append(chunk.dropna(subset=["vital_name"]))
 
full_vitals = pd.concat(vital_list, ignore_index=True)
del vital_list
gc.collect()
 
# =========================================================
# 2. AGGREGATE TO FLAT COLUMNS (Comprehensive Option C)
# =========================================================
print("Aggregating time-series to flat features...")

# Create the pivot
agg_vitals = full_vitals.groupby(["subject_id", "vital_name"])["valuenum"].agg(
    ["min", "max", "mean", "first"]
).unstack()
 
# Flatten the multi-index columns: e.g., ('min', 'heart_rate') -> 'heart_rate_min'
agg_vitals.columns = [f"{name}_{stat}" for stat, name in agg_vitals.columns]
agg_vitals = agg_vitals.reset_index()
 
# =========================================================
# 3. MERGE TO SPINE
# =========================================================
ml_matrix = SPINE.merge(agg_vitals, on="subject_id", how="left")

# Save the alignment matrix
output_path = "ml_cohort_with_vitals.csv"
ml_matrix.to_csv(output_path, index=False)
 
print(f"\n✔ Feature Extraction Complete.")
print(f"Final ML Matrix shape: {ml_matrix.shape}")
print(f"Matrix saved to: {output_path}")
 

Extracting and aggregating 24-hour Day-0 vitals...
Aggregating time-series to flat features...

✔ Feature Extraction Complete.
Final ML Matrix shape: (364626, 43)
Matrix saved to: ml_cohort_with_vitals.csv


In [1]:
import gc
import pandas as pd
from pathlib import Path
 
# =========================================================
# 0. CONFIGURATION & TARGET LAB DICTIONARY
# =========================================================
DATA_DIR = Path("../data/mimic-iv")
MATRIX_IN = "ml_cohort_with_vitals.csv"
MATRIX_OUT = "ml_cohort_master_v1.csv"
 
# Load the current ML spine (which now includes vitals)
ml_matrix = pd.read_csv(MATRIX_IN)
ml_matrix["index_time"] = pd.to_datetime(ml_matrix["index_time"])
 
# Comprehensive Lab Mapping (User Requested + Aortic Essentials)
# Note: MIMIC frequently has multiple ItemIDs for the same concept (e.g., Blood Gas vs Central Lab)
lab_ids = {
    # Essentials
    50915: "ddimer", 51003: "troponin_t", 51002: "troponin_i", 
    50889: "crp", 51214: "fibrinogen", 51237: "inr",
    # Renal & Electrolytes
    50912: "creatinine", 50983: "sodium", 50824: "sodium_bg",
    50971: "potassium", 50822: "potassium_bg", 50931: "glucose",
    50809: "glucose_bg", 51006: "bun", 50902: "chloride",
    50806: "chloride_bg", 50882: "bicarbonate", 50868: "anion_gap",
    50960: "magnesium", 50893: "calcium_total", 50808: "calcium_free",
    50970: "phosphate",
    # CBC & Hematology
    51222: "hemoglobin", 50811: "hemoglobin_bg", 51221: "hematocrit",
    50810: "hematocrit_bg", 51265: "platelets", 51301: "wbc", 
    51300: "wbc_count", 51249: "mchc", 51248: "mch", 51250: "mcv", 
    51277: "rdw", 51279: "rbc", 51244: "lymphocytes", 51200: "eosinophils",
    51256: "monocytes", 51146: "basophils", 51254: "neutrophils", 51274: "pt",
    # Blood Gas & Perfusion
    50820: "ph", 50821: "po2", 50813: "lactate",
    # Hepatic
    50861: "alt", 50878: "ast", 50885: "bilirubin_total", 
    50863: "alk_phos", 50862: "albumin"
}

# =========================================================
# 1. PROCESS LABS IN CHUNKS
# =========================================================

print("Extracting and aggregating 24-hour Day-0 Labs...")
lab_list = []

for chunk in pd.read_csv(DATA_DIR / "hosp" / "labevents.csv", 
                         usecols=["subject_id", "charttime", "itemid", "valuenum"], 
                         chunksize=3_000_000):
    # Filter to cohort and targeted labs
    chunk = chunk[
        (chunk["subject_id"].isin(ml_matrix["subject_id"])) & 
        (chunk["itemid"].isin(lab_ids.keys()))
    ].copy()
    if chunk.empty:
        continue
    chunk["charttime"] = pd.to_datetime(chunk["charttime"])
    # Map to cohort index times
    chunk = chunk.merge(ml_matrix[["subject_id", "index_time"]], on="subject_id", how="inner")
    # Enforce Day-0 Window: (0 <= time_diff <= 24 hours)
    delta = (chunk["charttime"] - chunk["index_time"]).dt.total_seconds() / 3600
    chunk = chunk[(delta >= 0) & (delta <= 24)].copy()
    # Clean the valuenum (drop text artifacts that slipped into the numeric column)
    chunk["valuenum"] = pd.to_numeric(chunk["valuenum"], errors="coerce")
    chunk.dropna(subset=["valuenum"], inplace=True)
    chunk["lab_name"] = chunk["itemid"].map(lab_ids)
    lab_list.append(chunk[["subject_id", "lab_name", "valuenum"]])
 
full_labs = pd.concat(lab_list, ignore_index=True)
del lab_list
gc.collect()
 
# =========================================================
# 2. AGGREGATE TO FLAT COLUMNS (Min, Max, Mean)
# =========================================================

print("Aggregating time-series to flat features...")
 
# Create the pivot
agg_labs = full_labs.groupby(["subject_id", "lab_name"])["valuenum"].agg(
    ["min", "max", "mean"]
).unstack()
 
# Flatten the multi-index columns: e.g., ('min', 'ddimer') -> 'ddimer_min'
agg_labs.columns = [f"{name}_{stat}" for stat, name in agg_labs.columns]
agg_labs = agg_labs.reset_index()
 
# =========================================================
# 3. MERGE TO MASTER SPINE
# =========================================================

final_matrix = ml_matrix.merge(agg_labs, on="subject_id", how="left")

final_matrix.to_csv(MATRIX_OUT, index=False)
 
print(f"\n✔ Day-0 Lab Extraction Complete.")
print(f"Final ML Matrix shape: {final_matrix.shape}")
print(f"Matrix saved to: {MATRIX_OUT}")
 

Extracting and aggregating 24-hour Day-0 Labs...
Aggregating time-series to flat features...

✔ Day-0 Lab Extraction Complete.
Final ML Matrix shape: (364626, 184)
Matrix saved to: ml_cohort_master_v1.csv


In [2]:
import pandas as pd
from pathlib import Path
 
# =========================================================
# 0. CONFIGURATION
# =========================================================

DATA_DIR = Path("../data/mimic-iv")
MATRIX_IN = "ml_cohort_master_v1.csv"
MATRIX_OUT = "ml_cohort_master_v2.csv" 
# Load the existing ML spine (contains age as 'index_age', vitals, and labs)
ml_matrix = pd.read_csv(MATRIX_IN)

print("--- Step 1: Extracting Static Demographics ---")
# Pull Gender and Anchor Year Group (Age is already present as 'index_age')
pts = pd.read_csv(
    DATA_DIR / "hosp" / "patients.csv", 
    usecols=["subject_id", "gender", "anchor_year_group"]
)
ml_matrix = ml_matrix.merge(pts, on="subject_id", how="left")

print("--- Step 2: Extracting Dynamic Demographics (Rescuing Standalone EDs) ---")
# Pull Race, Insurance, and Marital Status from the admissions table
adm_cols = ["subject_id", "admittime", "race", "insurance", "marital_status"]
adms = pd.read_csv(DATA_DIR / "hosp" / "admissions.csv", usecols=adm_cols)
adms["admittime"] = pd.to_datetime(adms["admittime"])
 
# Sort chronologically to ensure we grab the most recent demographic update per patient
adms_sorted = adms.sort_values(by=["subject_id", "admittime"])
 
# Group by patient and take the absolute LAST valid (non-null) observation for each category
latest_demographics = adms_sorted.groupby("subject_id")[["race", "insurance", "marital_status"]].last().reset_index()
 
# Merge onto the master matrix
ml_matrix = ml_matrix.merge(latest_demographics, on="subject_id", how="left")
 
# =========================================================
# 3. REORDER COLUMNS & EXPORT
# =========================================================
print("--- Step 3: Formatting and Exporting ---")

# Shift the new demographics to the front of the matrix (right after index_age) for readability
front_cols = [
    "subject_id", "index_hadm_id", "index_time", "is_aortic_dissection", 
    "admission_location", "encounter_urgency", 
    "index_age", "gender", "race", "insurance", "marital_status", "anchor_year_group"
]
# Get the remaining vital and lab columns dynamically
remaining_cols = [col for col in ml_matrix.columns if col not in front_cols]

ml_matrix = ml_matrix[front_cols + remaining_cols]
ml_matrix.to_csv(MATRIX_OUT, index=False)
 
print(f"✔ Demographics successfully merged.")
print(f"Final ML Matrix shape: {ml_matrix.shape}")
print(f"Matrix saved to: {MATRIX_OUT}")
 

--- Step 1: Extracting Static Demographics ---
--- Step 2: Extracting Dynamic Demographics (Rescuing Standalone EDs) ---
--- Step 3: Formatting and Exporting ---
✔ Demographics successfully merged.
Final ML Matrix shape: (364626, 189)
Matrix saved to: ml_cohort_master_v2.csv


In [2]:
import pandas as pd

master_df = pd.read_csv("./ml_cohort_master_v2.csv")

null_density = (master_df.isna().sum().sum() / master_df.size) * 100
print(f"Global Null Density: {null_density:.1f}%")

Global Null Density: 70.1%
